###### 133_4_light_finish_lcm_display_question.ipynb

In [28]:
import pandas as pd

In [29]:
df = pd.read_csv(
    "dataset/lcm_due_style_practice.csv",
)
display(df.head(5))

,no,site,model,defect_category,defect_count,due_date,status,priority,comment
0,1,GSE,LCM-A,点灯不良,2,2026-08-01,対応中,中,通常確認
1,2,WSE,LCM-B,外観不良,6,2026-08-03,未対応,高,期限注意
2,3,META,LCM-A,寸法不良,1,2026-08-10,完了,低,問題なし
3,4,MELAC,LCM-C,異音,8,2026-07-30,未対応,高,期限超過
4,5,GSE,LCM-B,点灯不良,0,2026-08-15,完了,低,再発なし


In [30]:
df["defect_count_raw"] = df["defect_count"]
df["due_date_raw"] = df["due_date"]

In [31]:
df[
    [
        "defect_count",
        "defect_count_raw",
        "due_date",
        "due_date_raw"
    ]
].dtypes

defect_count        object
defect_count_raw    object
due_date            object
due_date_raw        object
dtype: object

In [32]:
df["defect_count_clean"] = (
    df["defect_count"]
    .astype(str)
    .str.strip()
)
df["due_date_clean"] = (
    df["due_date"]
    .astype(str)
    .str.strip()
    .str.replace("/", "-", regex=False)
)

df[
    [
        "defect_count",
        "defect_count_clean",
        "due_date",
        "due_date_clean",
    ]
]

,defect_count,defect_count_clean,due_date,due_date_clean
0,2,2,2026-08-01,2026-08-01
1,6,6,2026-08-03,2026-08-03
2,1,1,2026-08-10,2026-08-10
3,8,8,2026-07-30,2026-07-30
4,0,0,2026-08-15,2026-08-15
5,3,3,2026/08/05,2026-08-05
6,未入力,未入力,2026-08-08,2026-08-08
7,5,5,未入力,未入力
8,7,7,NG_DATE,NG_DATE
9,2,2,2026-08-02,2026-08-02


In [33]:
df["defect_count_num"] = pd.to_numeric(
    df["defect_count_clean"],
    errors="coerce",
)
display(df["defect_count_num"].dtype)

df["due_date_dt"] = pd.to_datetime(
    df["due_date_clean"],
    errors="coerce",
)
display(df["due_date_dt"].dtypes)

dtype('float64')

dtype('<M8[ns]')

In [34]:
base_date = pd.Timestamp("2026-08-05")

In [35]:
base_date

Timestamp('2026-08-05 00:00:00')

In [36]:
df["is_count_ng"] = df["defect_count_num"].isna()
df["is_date_ng"] = df["due_date_dt"].isna()
df["is_overdue"] = df["due_date_dt"].notna() & (df["due_date_dt"] < base_date)
df["is_not_started"] = df["status"] == "未対応"
df["is_status_unknown"] = df["status"] == "不明"
df["is_high_priority"] = df["priority"] == "高"
df["is_many_defects"] = df["defect_count_num"] >= 5

In [38]:
def make_review_reason(row):
    reasons = []

    if row["is_count_ng"]:
        reasons.append("件数NG")

    if row["is_date_ng"]:
        reasons.append("日付NG")

    if row["is_overdue"]:
        reasons.append("期限超過")

    if row["is_not_started"]:
        reasons.append("未対応")

    if row["is_status_unknown"]:
        reasons.append("ステータス不明")

    if row["is_high_priority"]:
        reasons.append("高優先度")

    if row["is_many_defects"]:
        reasons.append("件数以上")

    return " / ".join(reasons)

df["review_reason"] = df.apply(
    make_review_reason,
    axis=1,
)

In [39]:
df["review_reason"]

0                        期限超過
1    期限超過 / 未対応 / 高優先度 / 件数以上
2                            
3    期限超過 / 未対応 / 高優先度 / 件数以上
4                            
5                            
6           件数NG / 未対応 / 高優先度
7                 日付NG / 件数以上
8          日付NG / 高優先度 / 件数以上
9              期限超過 / ステータス不明
Name: review_reason, dtype: object

In [40]:
#cell9
review_condition = (
    df["is_count_ng"]
    | df["is_date_ng"]
    | df["is_overdue"]
    | df["is_not_started"]
    | df["is_status_unknown"]
    | df["is_high_priority"]
    | df["is_many_defects"]
)

df_review = df[review_condition].copy()

In [41]:
df_review

,no,site,model,defect_category,defect_count,due_date,status,priority,comment,defect_count_raw,...,defect_count_num,due_date_dt,is_count_ng,is_date_ng,is_overdue,is_not_started,is_status_unknown,is_high_priority,is_many_defects,review_reason
0,1,GSE,LCM-A,点灯不良,2,2026-08-01,対応中,中,通常確認,2,...,2.0,2026-08-01,False,False,True,False,False,False,False,期限超過
1,2,WSE,LCM-B,外観不良,6,2026-08-03,未対応,高,期限注意,6,...,6.0,2026-08-03,False,False,True,True,False,True,True,期限超過 / 未対応 / 高優先度 / 件数以上
3,4,MELAC,LCM-C,異音,8,2026-07-30,未対応,高,期限超過,8,...,8.0,2026-07-30,False,False,True,True,False,True,True,期限超過 / 未対応 / 高優先度 / 件数以上
6,7,META,LCM-B,寸法不良,未入力,2026-08-08,未対応,高,件数未入力,未入力,...,NaN,2026-08-08,True,False,False,True,False,True,False,件数NG / 未対応 / 高優先度
7,8,MELAC,LCM-A,異音,5,未入力,対応中,中,期限未入力,5,...,5.0,NaT,False,True,False,False,False,False,True,日付NG / 件数以上
8,9,GSE,LCM-C,点灯不良,7,NG_DATE,完了,高,日付NG,7,...,7.0,NaT,False,True,False,False,False,True,True,日付NG / 高優先度 / 件数以上
9,10,WSE,LCM-A,外観不良,2,2026-08-02,不明,中,ステータス不明,2,...,2.0,2026-08-02,False,False,True,False,True,False,False,期限超過 / ステータス不明


In [43]:
display_cols = [
    "no",
    "site",
    "model",
    "defect_category",
    "defect_count",
    "due_date",
    "status",
    "priority",
    "comment",
    "review_reason",
]

df_review_display = df_review[display_cols].copy()

print("===確認対象データ全体===")
display(df_review_display)

===確認対象データ全体===


,no,site,model,defect_category,defect_count,due_date,status,priority,comment,review_reason
0,1,GSE,LCM-A,点灯不良,2,2026-08-01,対応中,中,通常確認,期限超過
1,2,WSE,LCM-B,外観不良,6,2026-08-03,未対応,高,期限注意,期限超過 / 未対応 / 高優先度 / 件数以上
3,4,MELAC,LCM-C,異音,8,2026-07-30,未対応,高,期限超過,期限超過 / 未対応 / 高優先度 / 件数以上
6,7,META,LCM-B,寸法不良,未入力,2026-08-08,未対応,高,件数未入力,件数NG / 未対応 / 高優先度
7,8,MELAC,LCM-A,異音,5,未入力,対応中,中,期限未入力,日付NG / 件数以上
8,9,GSE,LCM-C,点灯不良,7,NG_DATE,完了,高,日付NG,日付NG / 高優先度 / 件数以上
9,10,WSE,LCM-A,外観不良,2,2026-08-02,不明,中,ステータス不明,期限超過 / ステータス不明
